In [0]:
dbutils.widgets.text("loaded_date",defaultValue="2026-09-24")
dbutils.widgets.text("loaded_file",defaultValue="online_retail_II-1.csv")
loaded_date=dbutils.widgets.get("loaded_date")
loaded_file=dbutils.widgets.get("loaded_file")
dbutils.widgets.text('bronze_table','onlineretail_ws.bronze.online_retail')
bronze_table=dbutils.widgets.get("bronze_table")
dbutils.jobs.taskValues.set("bronze_table", bronze_table)


In [0]:
import pyspark.sql.functions as F

In [0]:
print(loaded_file)
print(loaded_date)


In [0]:
df = (
    spark.read
        .format("csv")
        .option("inferSchema", True)
        .option("header", True)
        .load(f"abfss://raw@onlineretailproject.dfs.core.windows.net/{loaded_date}/{loaded_file}")
)

In [0]:
df=df.withColumn("source_path",F.col("_metadata.file_path"))\
    .withColumn("source_file",F.col("_metadata.file_name"))\
    .withColumn("ingestion_time",F.current_timestamp())\
    .withColumnRenamed("Customer ID","Customer_ID")

In [0]:
df.write.format('delta') \
    .option('mergeSchema', 'true') \
    .mode('append') \
    .option("path", "abfss://bronze@onlineretailproject.dfs.core.windows.net/online_retail/") \
    .saveAsTable(bronze_table)